# LeetCode 962 — Maximum Width Ramp bằng Python

Link đề: https://leetcode.com/problems/maximum-width-ramp/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Stack (ngăn xếp), Monotonic Stack (ngăn xếp đơn điệu), Two Pointers (hai con trỏ), Sorting (sắp xếp)

Skeleton LeetCode:

```python
class Solution:
    def maxWidthRamp(self, nums: List[int]) -> int:
        ...
```

Ghi chú: một **dốc (ramp)** là cặp `(i, j)` với `i < j` và `nums[i] <= nums[j]`; **độ rộng** là `j - i`. Cần độ rộng lớn nhất. Quan sát then chốt: điểm bắt đầu `i` tối ưu **chỉ có thể là một "kỷ lục thấp mới"** (phần tử nhỏ hơn mọi phần tử đứng trước nó): nếu `nums[k] <= nums[i]` với `k < i` thì `k` luôn tốt hơn `i` (xa hơn mà vẫn thỏa). Điều đó dẫn tới **ngăn xếp đơn điệu giảm** các chỉ số bắt đầu, rồi quét `j` từ phải sang trái để "ăn" các chỉ số này.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm `j - i` lớn nhất với `i < j` và `nums[i] <= nums[j]`, hoặc `0` nếu không có.
2. Nắm các công cụ Python cần thiết: danh sách làm ngăn xếp, `sorted` với khóa, mảng min tiền tố / max hậu tố, hai con trỏ, và `bisect`.
3. Hiểu nguyên lý: ứng viên điểm bắt đầu là các **kỷ lục thấp** (prefix minima), tạo thành dãy giảm; với mỗi `j` ta cần chỉ số **sớm nhất** trong ứng viên có giá trị `<= nums[j]`.
4. Đi qua 4 cách: duyệt mọi cặp (brute), ngăn xếp đơn điệu + quét ngược (cách nộp chính), sắp xếp chỉ số theo giá trị rồi giữ chỉ số nhỏ nhất đã gặp, và hai con trỏ trên mảng min tiền tố/max hậu tố; đối chiếu bằng oracle.
5. Tránh các lỗi kinh điển: chỉ xét phần tử min và max toàn cục, dùng `<` thay `<=`, dựng ngăn xếp từ **mọi** phần tử, và quét ngược nhưng không `pop` nhiều lần.

## 2. Đề bài & ví dụ

Một **dốc** trong mảng số nguyên `nums` là cặp `(i, j)` với `i < j` và `nums[i] <= nums[j]`. **Độ rộng** của dốc là `j - i`.

Cho mảng `nums`, trả về **độ rộng lớn nhất** của một dốc; nếu không có dốc nào trả `0`.

**Ví dụ 1:** `nums = [6,0,8,2,1,5]` -> `4` (dốc `(1, 5)`: `0 <= 5`).

**Ví dụ 2:** `nums = [9,8,1,0,1,9,4,0,4,1]` -> `7` (dốc `(2, 9)`: `1 <= 1`).

**Ràng buộc:**

- `2 <= nums.length <= 5 * 10^4`
- `0 <= nums[i] <= 5 * 10^4`

In [1]:
from typing import List, Tuple
from bisect import bisect_left
import random, time

examples = [
    ([6, 0, 8, 2, 1, 5], 4),
    ([9, 8, 1, 0, 1, 9, 4, 0, 4, 1], 7),
]
for nums, expected in examples:
    print(f"nums={nums} kỳ vọng={expected}")

nums=[6, 0, 8, 2, 1, 5] kỳ vọng=4
nums=[9, 8, 1, 0, 1, 9, 4, 0, 4, 1] kỳ vọng=7


## 3. Kiến thức nền

### 3.1. Kỷ lục thấp (prefix minima)

Chỉ số `i` là **kỷ lục thấp** nếu `nums[i] < nums[k]` với mọi `k < i` (nhỏ hơn **nghiêm ngặt** mọi phần tử trước). Dãy giá trị của các kỷ lục thấp **giảm nghiêm ngặt**.

In [2]:
def record_lows(nums: List[int]) -> List[int]:
    stack = []
    for i, x in enumerate(nums):
        if not stack or x < nums[stack[-1]]:
            stack.append(i)
    return stack

for nums, _ in examples:
    r = record_lows(nums)
    print(nums, "-> kỷ lục thấp ở chỉ số", r, "giá trị", [nums[i] for i in r])

[6, 0, 8, 2, 1, 5] -> kỷ lục thấp ở chỉ số [0, 1] giá trị [6, 0]
[9, 8, 1, 0, 1, 9, 4, 0, 4, 1] -> kỷ lục thấp ở chỉ số [0, 1, 2, 3] giá trị [9, 8, 1, 0]


### 3.2. Vì sao chỉ cần xét kỷ lục thấp làm điểm bắt đầu

Nếu `i` không phải kỷ lục thấp thì tồn tại `k < i` với `nums[k] <= nums[i]`. Mọi dốc `(i, j)` (`nums[i] <= nums[j]`) khi đó cho dốc `(k, j)` **rộng hơn** (`j - k > j - i`) và hợp lệ (`nums[k] <= nums[i] <= nums[j]`). Vậy điểm bắt đầu tối ưu luôn là một kỷ lục thấp.

### 3.3. Quét ngược và "ăn" ngăn xếp

Duyệt `j` từ phải sang trái. Ngăn xếp các kỷ lục thấp có đỉnh là chỉ số **lớn nhất** (kỷ lục thấp gần nhất, giá trị nhỏ nhất). Nếu `nums[stack[-1]] <= nums[j]` thì dốc `(stack[-1], j)` hợp lệ; chỉ số `stack[-1]` **không bao giờ cần dùng lại** (các `j` nhỏ hơn cho dốc hẹp hơn với nó), nên `pop` và thử tiếp phần tử dưới đỉnh.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `[6, 0, 8, 2, 1, 5]`. Kỷ lục thấp: chỉ số `0` (giá trị `6`), chỉ số `1` (giá trị `0`). Ngăn xếp `[0, 1]`.

Quét `j` từ `5` xuống: `j = 5` (giá trị `5`): đỉnh `1` (giá trị `0 <= 5`): dốc `(1, 5)` rộng `4`, pop; đỉnh `0` (giá trị `6 > 5`) dừng. `j = 4, 3, 2`: đỉnh `0` giá trị `6`: `j = 2` có giá trị `8 >= 6`: dốc `(0, 2)` rộng `2`, pop. Đáp án `max(4, 2) = 4`.

## 5. Cách 1 — Brute Force: duyệt mọi cặp

**Ý tưởng.** Hai vòng lặp `i < j`, kiểm tra `nums[i] <= nums[j]` và cập nhật `j - i` lớn nhất. Đây là định nghĩa trực tiếp nên cũng là **oracle**.

In [3]:
def ramp_brute(nums: List[int]) -> int:
    n = len(nums)
    best = 0
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] <= nums[j]:
                best = max(best, j - i)
    return best

for nums, expected in examples:
    got = ramp_brute(nums)
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[6, 0, 8, 2, 1, 5] -> 4 OK
[9, 8, 1, 0, 1, 9, 4, 0, 4, 1] -> 7 OK


**Độ phức tạp và nút thắt.** `O(n^2)`: với `n = 5 * 10^4` là `~ 1,25 * 10^9` phép, quá chậm trong Python. Xem thời gian tăng bốn lần khi `n` tăng gấp đôi:

In [4]:
rng = random.Random(962)
for n in (500, 1000, 2000, 4000):
    nums = [rng.randint(0, 50000) for _ in range(n)]
    nums.sort(reverse=True)                                # giảm dần: không có dốc, phải duyệt hết
    t0 = time.perf_counter()
    ramp_brute(nums)
    print(f"n={n:5}: {1000 * (time.perf_counter() - t0):9.1f} ms")

n=  500:       6.4 ms
n= 1000:      27.4 ms
n= 2000:      96.4 ms


n= 4000:     377.5 ms


Nút thắt: với mỗi `i` ta quét lại mọi `j`; nhưng chỉ có **kỷ lục thấp** là ứng viên điểm bắt đầu và với mỗi `j` ta chỉ cần ứng viên **sớm nhất** thỏa điều kiện.

## 6. Ý tưởng thuật toán / bước đột phá

**Ứng viên điểm bắt đầu là các kỷ lục thấp** (dãy giảm), và với mỗi `j` ta cần chỉ số **nhỏ nhất** trong dãy đó có `nums[i] <= nums[j]`. Có nhiều cách tìm hiệu quả:

| Cách | Ý tưởng |
|---|---|
| 2 | Ngăn xếp các kỷ lục thấp, quét `j` từ phải sang trái và `pop` (nộp) |
| 3 | Sắp xếp chỉ số theo `(giá trị, chỉ số)`; với mỗi chỉ số theo thứ tự đó, giữ chỉ số nhỏ nhất đã gặp: mọi phần tử đã gặp có giá trị `<=` |
| 4 | Hai con trỏ: min tiền tố `lmin[i]` (không tăng) và max hậu tố `rmax[j]` (không tăng); dịch `j` khi còn `lmin[i] <= rmax[j]` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Ngăn xếp đơn điệu và quét ngược (cách nộp chính)

**Giải thích từng đoạn.**

- Xây `stack` các chỉ số kỷ lục thấp (từ trái sang phải: thêm `i` khi `nums[i] < nums[stack[-1]]`).
- Với `j` từ `n - 1` xuống `0`: `while stack and nums[stack[-1]] <= nums[j]: ans = max(ans, j - stack.pop())`.
- Dừng sớm: nếu `ans >= j` thì không thể cải thiện (độ rộng tối đa còn lại `j`); có thể `break` khi `stack` rỗng.

In [5]:
def ramp_stack(nums: List[int]) -> int:
    stack = []
    for i, x in enumerate(nums):
        if not stack or x < nums[stack[-1]]:
            stack.append(i)
    ans = 0
    for j in range(len(nums) - 1, -1, -1):
        while stack and nums[stack[-1]] <= nums[j]:
            ans = max(ans, j - stack.pop())
        if not stack:
            break
    return ans

for nums, expected in examples:
    got = ramp_stack(nums)
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[6, 0, 8, 2, 1, 5] -> 4 OK
[9, 8, 1, 0, 1, 9, 4, 0, 4, 1] -> 7 OK


**Code nộp LeetCode (Cách 2):**

In [6]:
class Solution:
    def maxWidthRamp(self, nums: List[int]) -> int:
        stack = []
        for i, x in enumerate(nums):
            if not stack or x < nums[stack[-1]]:
                stack.append(i)
        ans = 0
        for j in range(len(nums) - 1, -1, -1):
            while stack and nums[stack[-1]] <= nums[j]:
                ans = max(ans, j - stack.pop())
            if not stack:
                break
        return ans

sol_stack = Solution().maxWidthRamp
assert [sol_stack(list(n)) for n, _ in examples] == [e for _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n)` thời gian (mỗi chỉ số vào và ra ngăn xếp tối đa một lần), `O(n)` bộ nhớ.

---

### 7.2. Cách 3 — Sắp xếp chỉ số theo giá trị

**Ý tưởng khác.** Sắp xếp các chỉ số theo `(nums[i], i)` tăng dần. Duyệt theo thứ tự đó, giữ `min_index` là chỉ số **nhỏ nhất** trong các chỉ số đã duyệt (mọi phần tử đã duyệt có giá trị `<=` phần tử hiện tại). Với chỉ số hiện tại `j`: nếu `min_index < j` thì dốc `(min_index, j)` hợp lệ: `ans = max(ans, j - min_index)`; rồi `min_index = min(min_index, j)`.

**Giải thích từng đoạn.** Khi xử lý `j`, tập chỉ số đã duyệt gồm mọi phần tử có `(giá trị, chỉ số)` nhỏ hơn, tức giá trị `<= nums[j]`; trong đó phần tử có chỉ số nhỏ nhất cho dốc rộng nhất kết thúc ở `j`.

In [7]:
def ramp_sort(nums: List[int]) -> int:
    order = sorted(range(len(nums)), key=lambda i: (nums[i], i))
    min_index = len(nums)
    ans = 0
    for j in order:
        ans = max(ans, j - min_index)              # nếu min_index > j thì giá trị âm, không ảnh hưởng vì ans >= 0
        min_index = min(min_index, j)
    return ans

for nums, expected in examples:
    got = ramp_sort(nums)
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[6, 0, 8, 2, 1, 5] -> 4 OK
[9, 8, 1, 0, 1, 9, 4, 0, 4, 1] -> 7 OK


**Code nộp LeetCode (Cách 3):**

In [8]:
class Solution:
    def maxWidthRamp(self, nums: List[int]) -> int:
        order = sorted(range(len(nums)), key=lambda i: (nums[i], i))
        min_index = len(nums)
        ans = 0
        for j in order:
            ans = max(ans, j - min_index)
            min_index = min(min_index, j)
        return ans

sol_sort = Solution().maxWidthRamp
assert [sol_sort(list(n)) for n, _ in examples] == [e for _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n log n)` thời gian (sắp xếp), `O(n)` bộ nhớ.

---

### 7.3. Cách 4 — Hai con trỏ trên min tiền tố và max hậu tố

**Ý tưởng khác.** `lmin[i] = min(nums[0..i])` (không tăng), `rmax[j] = max(nums[j..n-1])` (không tăng). Điều kiện "tồn tại dốc `(i', j')` với `i' <= i`, `j' >= j`" ⇔ `lmin[i] <= rmax[j]`. Dùng hai con trỏ `i`, `j` cùng bắt đầu từ `0`: nếu `lmin[i] <= rmax[j]` thì ghi nhận độ rộng `j - i` và tăng `j`; ngược lại tăng `i`.

**Giải thích từng đoạn.** Mỗi bước tăng đúng một con trỏ nên vòng lặp tối đa `2n` bước. Với mỗi `i`, `j` chỉ tiến (không lùi) vì `rmax` không tăng nên mỗi `i` lớn hơn có `j` tối đa không lớn hơn.

In [9]:
def ramp_two_pointers(nums: List[int]) -> int:
    n = len(nums)
    lmin = nums[:]
    for i in range(1, n):
        lmin[i] = min(lmin[i - 1], nums[i])
    rmax = nums[:]
    for j in range(n - 2, -1, -1):
        rmax[j] = max(rmax[j + 1], nums[j])
    ans = i = j = 0
    while j < n:
        while i < j and lmin[i] > rmax[j]:
            i += 1
        ans = max(ans, j - i)
        j += 1
    return ans

for nums, expected in examples:
    got = ramp_two_pointers(nums)
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[6, 0, 8, 2, 1, 5] -> 4 OK
[9, 8, 1, 0, 1, 9, 4, 0, 4, 1] -> 7 OK


**Code nộp LeetCode (Cách 4):**

In [10]:
class Solution:
    def maxWidthRamp(self, nums: List[int]) -> int:
        n = len(nums)
        lmin = nums[:]
        for i in range(1, n):
            lmin[i] = min(lmin[i - 1], nums[i])
        rmax = nums[:]
        for j in range(n - 2, -1, -1):
            rmax[j] = max(rmax[j + 1], nums[j])
        ans = i = j = 0
        while j < n:
            while i < j and lmin[i] > rmax[j]:
                i += 1
            ans = max(ans, j - i)
            j += 1
        return ans

sol_two = Solution().maxWidthRamp
assert [sol_two(list(n)) for n, _ in examples] == [e for _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 2: `[9,8,1,0,1,9,4,0,4,1]`

In [11]:
def trace_stack(nums):
    stack = record_lows(nums)
    print("ngăn xếp các kỷ lục thấp (chỉ số):", stack, "giá trị:", [nums[i] for i in stack])
    ans = 0
    for j in range(len(nums) - 1, -1, -1):
        popped = []
        while stack and nums[stack[-1]] <= nums[j]:
            i = stack.pop()
            ans = max(ans, j - i)
            popped.append(i)
        print(f"j={j} (giá trị {nums[j]}): pop {popped} -> ans={ans}, ngăn xếp {stack}")
        if not stack:
            break
    return ans

assert trace_stack([9, 8, 1, 0, 1, 9, 4, 0, 4, 1]) == 7

ngăn xếp các kỷ lục thấp (chỉ số): [0, 1, 2, 3] giá trị: [9, 8, 1, 0]
j=9 (giá trị 1): pop [3, 2] -> ans=7, ngăn xếp [0, 1]
j=8 (giá trị 4): pop [] -> ans=7, ngăn xếp [0, 1]
j=7 (giá trị 0): pop [] -> ans=7, ngăn xếp [0, 1]
j=6 (giá trị 4): pop [] -> ans=7, ngăn xếp [0, 1]
j=5 (giá trị 9): pop [1, 0] -> ans=7, ngăn xếp []


### 8.2. Cách 3 cho ví dụ 1 (sắp xếp chỉ số)

`nums = [6, 0, 8, 2, 1, 5]`; thứ tự chỉ số theo `(giá trị, chỉ số)`: `1 (0), 4 (1), 3 (2), 5 (5), 0 (6), 2 (8)`.

In [12]:
def trace_sort(nums):
    order = sorted(range(len(nums)), key=lambda i: (nums[i], i))
    min_index, ans = len(nums), 0
    print("thứ tự chỉ số:", order)
    for j in order:
        width = j - min_index
        ans = max(ans, width)
        min_index = min(min_index, j)
        print(f"chỉ số {j} (giá trị {nums[j]}): độ rộng với min_index cũ = {width}, ans = {ans}, min_index mới = {min_index}")
    return ans

assert trace_sort([6, 0, 8, 2, 1, 5]) == 4

thứ tự chỉ số: [1, 4, 3, 5, 0, 2]
chỉ số 1 (giá trị 0): độ rộng với min_index cũ = -5, ans = 0, min_index mới = 1
chỉ số 4 (giá trị 1): độ rộng với min_index cũ = 3, ans = 3, min_index mới = 1
chỉ số 3 (giá trị 2): độ rộng với min_index cũ = 2, ans = 3, min_index mới = 1
chỉ số 5 (giá trị 5): độ rộng với min_index cũ = 4, ans = 4, min_index mới = 1
chỉ số 0 (giá trị 6): độ rộng với min_index cũ = -1, ans = 4, min_index mới = 0
chỉ số 2 (giá trị 8): độ rộng với min_index cũ = 2, ans = 4, min_index mới = 0


### 8.3. Cách 4 cho ví dụ 1 (hai con trỏ)

`lmin = [6, 0, 0, 0, 0, 0]`, `rmax = [8, 8, 8, 5, 5, 5]`. `j = 0..5` với `i` tăng khi `lmin[i] > rmax[j]`: ví dụ `j = 5`: `i = 0`: `lmin[0] = 6 > rmax[5] = 5` nên `i = 1`: `0 <= 5`: độ rộng `4`.

In [13]:
nums = [6, 0, 8, 2, 1, 5]
lmin, rmax = nums[:], nums[:]
for i in range(1, 6):
    lmin[i] = min(lmin[i - 1], nums[i])
for j in range(4, -1, -1):
    rmax[j] = max(rmax[j + 1], nums[j])
print("lmin =", lmin, "| rmax =", rmax)
assert ramp_two_pointers(nums) == 4

lmin = [6, 0, 0, 0, 0, 0] | rmax = [8, 8, 8, 5, 5, 5]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (ngăn xếp đơn điệu + quét ngược): `O(n)` thời gian, ý tưởng rõ ràng.

```python
class Solution:
    def maxWidthRamp(self, nums: List[int]) -> int:
        stack = []
        for i, x in enumerate(nums):
            if not stack or x < nums[stack[-1]]:
                stack.append(i)
        ans = 0
        for j in range(len(nums) - 1, -1, -1):
            while stack and nums[stack[-1]] <= nums[j]:
                ans = max(ans, j - stack.pop())
            if not stack:
                break
        return ans
```

In [14]:
class Solution:
    def maxWidthRamp(self, nums: List[int]) -> int:
        stack = []
        for i, x in enumerate(nums):
            if not stack or x < nums[stack[-1]]:
                stack.append(i)
        ans = 0
        for j in range(len(nums) - 1, -1, -1):
            while stack and nums[stack[-1]] <= nums[j]:
                ans = max(ans, j - stack.pop())
            if not stack:
                break
        return ans

sol = Solution()
for nums, expected in examples:
    got = sol.maxWidthRamp(list(nums))
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[6, 0, 8, 2, 1, 5] -> 4 OK
[9, 8, 1, 0, 1, 9, 4, 0, 4, 1] -> 7 OK


## 10. Vì sao đúng

**(a) Điểm bắt đầu là kỷ lục thấp.** Nếu `i` không phải kỷ lục thấp tồn tại `k < i` với `nums[k] <= nums[i]`; mọi dốc `(i, j)` bị thay bởi dốc rộng hơn `(k, j)`. Do đó đáp án bằng độ rộng lớn nhất của dốc có điểm bắt đầu trong tập kỷ lục thấp.

**(b) Cách 2 (ngăn xếp).** Các kỷ lục thấp theo thứ tự chỉ số tăng có giá trị **giảm nghiêm ngặt**. Khi quét `j` giảm dần, với đỉnh `t = stack[-1]` (chỉ số lớn nhất còn lại, giá trị nhỏ nhất còn lại): nếu `nums[t] <= nums[j]` thì dốc `(t, j)` hợp lệ. Chỉ số `t` không cần dùng cho `j' < j` vì `j' - t < j - t`. Nên `pop` là an toàn. Nếu `nums[t] > nums[j]` thì **mọi** phần tử dưới đỉnh có giá trị lớn hơn `nums[t]` (dãy giảm nghiêm ngặt từ dưới lên đỉnh) nên cũng `> nums[j]`: không còn dốc nào kết thúc ở `j` với điểm bắt đầu trong ngăn xếp. Với mỗi `j` và mỗi kỷ lục thấp `i` thỏa `nums[i] <= nums[j]`, `i` bị pop tại một `j'' >= j` có độ rộng `>= j - i`: giá trị `ans` bao quát mọi dốc tối ưu (theo (a)). `break` khi ngăn xếp rỗng là an toàn.

**(c) Cách 3.** Duyệt theo `(giá trị, chỉ số)` tăng: khi tới `j`, tập đã duyệt chính xác là các phần tử có giá trị `< nums[j]`, hoặc giá trị bằng với chỉ số nhỏ hơn, tức giá trị `<= nums[j]` và (nếu bằng thì chỉ số `< j`; phần tử cùng giá trị chỉ số lớn hơn chưa duyệt nhưng cũng không là điểm bắt đầu hợp lệ của `j` vì `i < j` bắt buộc; ta xử lý chúng ở lượt của chúng). Phần tử có chỉ số nhỏ nhất `min_index` trong tập đã duyệt tạo dốc rộng nhất kết thúc ở `j` nếu `min_index < j`; nếu `min_index > j` thì `j - min_index < 0` không làm tăng `ans >= 0`. Một dốc `(i, j)` bất kỳ có `(nums[i], i)` đứng trước `(nums[j], j)` trong thứ tự sắp xếp (vì `nums[i] <= nums[j]` và `i < j`), nên được xét.

**(d) Cách 4.** `lmin[i] <= rmax[j]` ⇔ tồn tại `i' <= i`, `j' >= j` với `nums[i'] <= nums[j']`... Hai con trỏ: với `j` tăng, `rmax[j]` không tăng nên điều kiện `lmin[i] <= rmax[j]` khó thỏa hơn, và `i` tối thiểu thỏa điều kiện không giảm khi `j` tăng; do đó `i` chỉ cần tiến. Kết quả `max(j - i)` bằng độ rộng dốc lớn nhất vì dốc rộng nhất `(i*, j*)` có `lmin[i*] <= nums[i*] <= nums[j*] <= rmax[j*]` và mọi `(i, j)` thỏa `lmin[i] <= rmax[j]` ứng với một dốc `(i', j')` với `i' <= i`, `j' >= j` rộng hơn hoặc bằng.

**Kiểm chứng bằng máy.** Oracle **độc lập**: duyệt mọi cặp `(i, j)` theo định nghĩa (Cách 1). So mọi cách với oracle trên mảng ngẫu nhiên, mảng nhiều giá trị trùng, mảng đơn điệu và mảng lớn (`n = 5 * 10^4`, với mẫu con cho oracle `O(n^2)` và so sánh chéo các cách `O(n)`/`O(n log n)` trên mảng lớn).

In [15]:
rng = random.Random(1962)
for _ in range(500):
    n = rng.randint(2, 30)
    nums = [rng.randint(0, rng.choice([2, 5, 50])) for _ in range(n)]
    b = ramp_brute(nums)
    assert b == ramp_stack(nums) == ramp_sort(nums) == ramp_two_pointers(nums), nums
print("Cách 1-4 khớp nhau trên 500 mảng ngẫu nhiên nhỏ")

Cách 1-4 khớp nhau trên 500 mảng ngẫu nhiên nhỏ


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp | `O(n^2)` | `O(1)` |
| 2. Ngăn xếp đơn điệu | `O(n)` | `O(n)` |
| 3. Sắp xếp chỉ số | `O(n log n)` | `O(n)` |
| 4. Hai con trỏ | `O(n)` | `O(n)` |

Đo thử với `n = 5 * 10^4` (cỡ tối đa):

In [16]:
rng = random.Random(11)
n = 5 * 10 ** 4
datasets = {
    "ngẫu nhiên": [rng.randint(0, 50000) for _ in range(n)],
    "giảm dần (đáp án 0 hoặc rất nhỏ)": sorted((rng.randint(0, 50000) for _ in range(n)), reverse=True),
    "tăng dần": sorted(rng.randint(0, 50000) for _ in range(n)),
    "toàn bằng nhau": [7] * n,
}
for dname, nums in datasets.items():
    ref = ramp_two_pointers(nums)
    row = []
    for name, fn in (("ngăn xếp", sol_stack), ("sắp xếp", sol_sort), ("hai con trỏ", sol_two), ("nộp", Solution().maxWidthRamp)):
        t0 = time.perf_counter()
        got = fn(list(nums))
        row.append(f"{name}: {1000 * (time.perf_counter() - t0):6.1f} ms")
        assert got == ref, (dname, name)
    print(f"{dname:34} đáp án {ref:5} | " + " | ".join(row))

ngẫu nhiên                         đáp án 49998 | ngăn xếp:    2.5 ms | sắp xếp:   35.7 ms | hai con trỏ:   20.2 ms | nộp:    4.5 ms


giảm dần (đáp án 0 hoặc rất nhỏ)   đáp án     7 | ngăn xếp:   16.6 ms | sắp xếp:   31.8 ms | hai con trỏ:   26.8 ms | nộp:   16.3 ms
tăng dần                           đáp án 49999 | ngăn xếp:    4.3 ms | sắp xếp:   23.9 ms | hai con trỏ:   21.1 ms | nộp:    4.9 ms


toàn bằng nhau                     đáp án 49999 | ngăn xếp:    3.8 ms | sắp xếp:   50.1 ms | hai con trỏ:   19.1 ms | nộp:    4.1 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [17]:
def solutions_list():
    return {
        "stack": ramp_stack,
        "sort": ramp_sort,
        "two": ramp_two_pointers,
        "sol_stack": sol_stack,
        "sol_sort": sol_sort,
        "sol_two": sol_two,
        "final": Solution().maxWidthRamp,
    }

def run_all(nums, expected=None, brute_limit=3000):
    ref = ramp_brute(nums) if len(nums) <= brute_limit else ramp_two_pointers(nums)
    for name, fn in solutions_list().items():
        assert fn(list(nums)) == ref, (name, nums[:20])
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", nums, expected, ref)
    return ref

# 12.1 ví dụ đề
for nums, expected in examples:
    assert run_all(nums, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1, 2], 1),                                 # hai phần tử tăng
    ([2, 1], 0),                                 # hai phần tử giảm: không có dốc
    ([5, 5], 1),                                 # bằng nhau: <= cho phép
    ([3, 2, 1], 0),                              # giảm dần
    ([1, 2, 3, 4, 5], 4),                        # tăng dần: cả mảng
    ([7, 7, 7, 7], 3),                           # toàn bằng nhau
    ([6, 0, 8, 2, 1, 5], 4),
    ([9, 8, 1, 0, 1, 9, 4, 0, 4, 1], 7),
    ([0, 50000], 1),                             # biên
    ([50000, 0], 0),
    ([2, 2, 1, 1, 2], 4),                        # (0, 4): 2 <= 2
    ([5, 1, 4, 2, 3], 3),                        # (1, 4): 1 <= 3 rộng 3
    ([4, 3, 2, 1, 5], 4),                        # (0, 4)
    ([1, 5, 0, 4, 2, 3], 5),                     # (0, 5): 1 <= 3 rộng 5
]
for nums, expected in edge:
    run_all(nums, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 14 ca


In [18]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_global_min_max(nums):                  # chỉ xét vị trí min toàn cục và max toàn cục
    i = nums.index(min(nums))
    j = len(nums) - 1 - nums[::-1].index(max(nums))
    return max(0, j - i) if nums[i] <= nums[j] and i < j else 0

def wrong_strict(nums):                          # dùng < thay <=
    stack = []
    for i, x in enumerate(nums):
        if not stack or x < nums[stack[-1]]:
            stack.append(i)
    ans = 0
    for j in range(len(nums) - 1, -1, -1):
        while stack and nums[stack[-1]] < nums[j]:
            ans = max(ans, j - stack.pop())
    return ans

def wrong_stack_all(nums):                       # ngăn xếp chứa mọi chỉ số (không chỉ kỷ lục thấp)
    stack = list(range(len(nums)))
    ans = 0
    for j in range(len(nums) - 1, -1, -1):
        while stack and nums[stack[-1]] <= nums[j]:
            ans = max(ans, j - stack.pop())
    return ans

def wrong_no_pop_loop(nums):                     # chỉ thử một ứng viên (if thay while)
    stack = []
    for i, x in enumerate(nums):
        if not stack or x < nums[stack[-1]]:
            stack.append(i)
    ans = 0
    for j in range(len(nums) - 1, -1, -1):
        if stack and nums[stack[-1]] <= nums[j]:
            ans = max(ans, j - stack.pop())
    return ans

def find_counter(wrong, tries=20000, seed=5):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 9)
        nums = [rng.randint(0, 6) for _ in range(n)]
        if wrong(list(nums)) != ramp_brute(nums):
            return nums
    return None

counters = {}
for name, fn in (("chỉ xét min và max toàn cục", wrong_global_min_max), ("dùng < thay <=", wrong_strict),
                 ("ngăn xếp chứa mọi chỉ số", wrong_stack_all), ("if thay while", wrong_no_pop_loop)):
    nums = find_counter(fn)
    assert nums is not None, name
    counters[name] = nums
    print(f"{name:30} nums={nums}: sai -> {fn(list(nums))}, đúng -> {ramp_brute(nums)}")
    run_all(nums)
print("12.3 phản ví dụ: OK")

chỉ xét min và max toàn cục    nums=[5, 2, 6, 5, 6, 5]: sai -> 3, đúng -> 5
dùng < thay <=                 nums=[5, 2, 6, 5, 6, 5]: sai -> 4, đúng -> 5
ngăn xếp chứa mọi chỉ số       nums=[5, 2, 6, 5, 6, 5]: sai -> 4, đúng -> 5
if thay while                  nums=[5, 2, 6, 5, 6, 5]: sai -> 4, đúng -> 5
12.3 phản ví dụ: OK


In [19]:
# 12.4 ngẫu nhiên nhỏ: n <= 40, nhiều dải giá trị (nhiều trùng lặp)
rng = random.Random(4962)
for _ in range(2000):
    n = rng.randint(2, 40)
    nums = [rng.randint(0, rng.choice([1, 3, 10, 50000])) for _ in range(n)]
    run_all(nums)
print("12.4 ngẫu nhiên n <= 40 (2000 ca): OK")

# 12.5 trung bình: n tới 1500 so với oracle O(n^2)
rng = random.Random(5962)
for _ in range(60):
    n = rng.randint(41, 1500)
    nums = [rng.randint(0, rng.choice([5, 100, 50000])) for _ in range(n)]
    run_all(nums)
print("12.5 ngẫu nhiên n tới 1500 (60 ca): OK")

# 12.6 lớn n = 5 * 10^4: nhiều dạng; oracle O(n^2) trên mẫu con 2000, các cách khớp nhau trên toàn mảng
rng = random.Random(6962)
n = 5 * 10 ** 4
big = {
    "ngẫu nhiên": [rng.randint(0, 50000) for _ in range(n)],
    "giảm dần": sorted((rng.randint(0, 50000) for _ in range(n)), reverse=True),
    "tăng dần": sorted(rng.randint(0, 50000) for _ in range(n)),
    "toàn bằng nhau": [7] * n,
    "chỉ vài giá trị": [rng.randint(0, 3) for _ in range(n)],
    "tăng rồi giảm": list(range(25000)) + list(range(25000, 0, -1)),
    "xen kẽ cao thấp": [50000 if i % 2 == 0 else 0 for i in range(n)],
}
expected_big = {"tăng dần": n - 1, "toàn bằng nhau": n - 1}
for name, nums in big.items():
    assert len(nums) <= n and all(0 <= x <= 50000 for x in nums), name
    ref = ramp_two_pointers(nums)
    for fname, fn in solutions_list().items():
        assert fn(list(nums)) == ref, (name, fname)
    sub = nums[:2000]
    assert Solution().maxWidthRamp(list(sub)) == ramp_brute(sub), name
    if name in expected_big:
        assert ref == expected_big[name], (name, ref)
    print(f"  {name:18} đáp án {ref}")
print("12.6 n = 5 * 10^4: OK")

# 12.7 tính chất: thêm phần tử lớn nhất ở cuối làm đáp án >= (n - 1 - chỉ số phần tử nhỏ nhất...); đáp án không đổi khi cộng hằng số; đảo thứ tự và đổi dấu tương ứng
rng = random.Random(7962)
for _ in range(300):
    n = rng.randint(2, 30)
    nums = [rng.randint(0, 100) for _ in range(n)]
    base = Solution().maxWidthRamp(list(nums))
    k = rng.randint(0, 50)
    assert Solution().maxWidthRamp([x + k for x in nums]) == base
    # đảo mảng và đổi dấu: dốc (i, j) với nums[i] <= nums[j] tương ứng dốc (n-1-j, n-1-i) với -nums[j] <= -nums[i]
    assert Solution().maxWidthRamp([-x + 100 for x in nums[::-1]]) == base
    assert Solution().maxWidthRamp(nums + [max(nums)]) >= base
print("12.7 tính chất (cộng hằng số, đối xứng đảo + đổi dấu, thêm phần tử): OK")

12.4 ngẫu nhiên n <= 40 (2000 ca): OK


12.5 ngẫu nhiên n tới 1500 (60 ca): OK


  ngẫu nhiên         đáp án 49998
  giảm dần           đáp án 6


  tăng dần           đáp án 49999
  toàn bằng nhau     đáp án 49999


  chỉ vài giá trị    đáp án 49999
  tăng rồi giảm      đáp án 49999


  xen kẽ cao thấp    đáp án 49998
12.6 n = 5 * 10^4: OK
12.7 tính chất (cộng hằng số, đối xứng đảo + đổi dấu, thêm phần tử): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét vị trí min và max toàn cục

Dốc rộng nhất không nhất thiết đi từ phần tử nhỏ nhất tới phần tử lớn nhất.

In [20]:
nums = counters["chỉ xét min và max toàn cục"]
print("nums =", nums, "| sai ->", wrong_global_min_max(list(nums)), "| đúng ->", Solution().maxWidthRamp(list(nums)))
assert wrong_global_min_max(list(nums)) != Solution().maxWidthRamp(list(nums))

nums = [5, 2, 6, 5, 6, 5] | sai -> 3 | đúng -> 5


Vì sao sai: ví dụ 2 có đáp án `7` ở `(2, 9)` với giá trị `1` và `1`, không phải min và max. **Đúng:** xét mọi kỷ lục thấp làm điểm bắt đầu.

### Lỗi 2 — Dùng `<` thay `<=`

Đề cho phép `nums[i] <= nums[j]` (bằng nhau vẫn là dốc).

In [21]:
nums = counters["dùng < thay <="]
print("nums =", nums, "| sai ->", wrong_strict(list(nums)), "| đúng ->", Solution().maxWidthRamp(list(nums)))
assert wrong_strict(list(nums)) != Solution().maxWidthRamp(list(nums))
print("ví dụ 2:", wrong_strict(examples[1][0]), "(sai, thiếu dốc (2, 9)) so với", Solution().maxWidthRamp(list(examples[1][0])))

nums = [5, 2, 6, 5, 6, 5] | sai -> 4 | đúng -> 5
ví dụ 2: 6 (sai, thiếu dốc (2, 9)) so với 7


Vì sao sai: bỏ qua các dốc có hai đầu bằng nhau. **Đúng:** `nums[stack[-1]] <= nums[j]`.

### Lỗi 3 — Ngăn xếp chứa mọi chỉ số

Chỉ các **kỷ lục thấp** mới là ứng viên bắt đầu; đẩy mọi chỉ số làm ngăn xếp mất tính đơn điệu, nên `pop` ở đỉnh bỏ sót các chỉ số sâu hơn.

In [22]:
nums = counters["ngăn xếp chứa mọi chỉ số"]
print("nums =", nums, "| sai ->", wrong_stack_all(list(nums)), "| đúng ->", Solution().maxWidthRamp(list(nums)))
assert wrong_stack_all(list(nums)) != Solution().maxWidthRamp(list(nums))

nums = [5, 2, 6, 5, 6, 5] | sai -> 4 | đúng -> 5


Vì sao sai: nếu đỉnh có giá trị lớn thì phần tử sâu hơn có giá trị nhỏ nhưng không được xét. **Đúng:** chỉ đẩy khi `nums[i] < nums[stack[-1]]`.

### Lỗi 4 — Dùng `if` thay cho `while` khi `pop`

Một `j` có thể "ăn" **nhiều** ứng viên liên tiếp (các chỉ số dưới đỉnh cũng thỏa).

In [23]:
nums = counters["if thay while"]
print("nums =", nums, "| sai ->", wrong_no_pop_loop(list(nums)), "| đúng ->", Solution().maxWidthRamp(list(nums)))
assert wrong_no_pop_loop(list(nums)) != Solution().maxWidthRamp(list(nums))

nums = [5, 2, 6, 5, 6, 5] | sai -> 4 | đúng -> 5


Vì sao sai: sau khi `pop` đỉnh, ứng viên kế tiếp (xa hơn về bên trái) có thể cũng thỏa với cùng `j` và cho dốc rộng hơn. **Đúng:** `while stack and nums[stack[-1]] <= nums[j]`.

### Lỗi 5 — Duyệt mọi cặp (`O(n^2)`)

Với `n = 5 * 10^4` là `~ 10^9` phép: TLE (xem mục 5).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "khoảng cách lớn nhất giữa `i < j` thỏa quan hệ giá trị" ⇒ **ngăn xếp đơn điệu** (điểm bắt đầu là kỷ lục thấp) + quét ngược, hoặc **sắp xếp** + chỉ số nhỏ nhất.
- Chỉ xét kỷ lục thấp (prefix minima) cho `i`, và kỷ lục cao hậu tố (suffix maxima) cho `j`.
- Quét `j` từ phải sang trái và `pop` mọi ứng viên thỏa: mỗi chỉ số vào/ra ngăn xếp một lần.
- Cùng họ: Next Greater Element (496, 503), Daily Temperatures (739), Largest Rectangle in Histogram (84), Sum of Subarray Minimums (907), Maximum Width of Binary Tree (662, khác chủ đề).
- Test nên có: giảm dần, tăng dần, toàn bằng nhau, nhiều trùng lặp, `n = 5 * 10^4`.

## 15. Biến thể

1. **Trả về chính cặp `(i, j)`** (bài tập 5): nhiều đáp án khi có nhiều dốc cùng độ rộng.
2. **Độ rộng lớn nhất với `nums[i] < nums[j]`** (nghiêm ngặt): đổi `<=` thành `<` và điều kiện kỷ lục thấp thành `<=`.
3. **Cặp `i < j` với `nums[i] >= nums[j]`:** đối xứng (đổi dấu mảng).
4. Bài liên quan: 456 (132 Pattern), 739.

Dưới đây là biến thể 2 (nghiêm ngặt) kiểm bằng vét cạn.

In [24]:
def ramp_strict(nums: List[int]) -> int:
    stack = []
    for i, x in enumerate(nums):
        if not stack or x < nums[stack[-1]]:           # vẫn là kỷ lục thấp nghiêm ngặt (giá trị bằng đứng sau không tốt hơn)
            stack.append(i)
    ans = 0
    for j in range(len(nums) - 1, -1, -1):
        while stack and nums[stack[-1]] < nums[j]:
            ans = max(ans, j - stack.pop())
    return ans

def ramp_strict_brute(nums):
    return max([j - i for i in range(len(nums)) for j in range(i + 1, len(nums)) if nums[i] < nums[j]], default=0)

rng = random.Random(15)
for _ in range(400):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(2, 20))]
    assert ramp_strict(nums) == ramp_strict_brute(nums), nums
print("biến thể nghiêm ngặt khớp vét cạn; ví dụ 2:", ramp_strict(examples[1][0]))

biến thể nghiêm ngặt khớp vét cạn; ví dụ 2: 6


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | Định nghĩa | `O(n^2)` | `O(1)` | Oracle |
| 2. Ngăn xếp đơn điệu | Kỷ lục thấp + quét ngược | `O(n)` | `O(n)` | **Cách nộp chính** |
| 3. Sắp xếp chỉ số | Chỉ số nhỏ nhất đã gặp | `O(n log n)` | `O(n)` | Ngắn gọn, ít sai |
| 4. Hai con trỏ | min tiền tố / max hậu tố | `O(n)` | `O(n)` | Không dùng ngăn xếp |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(record_lows([6, 0, 8, 2, 1, 5]))
print(record_lows([1, 2, 3]))
print(sorted(range(4), key=lambda i: ([3, 1, 3, 2][i], i)))
```

In [25]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `nums = [5, 3, 4, 1, 2, 6]`: ghi độ rộng lớn nhất và cặp `(i, j)` đạt được.

In [26]:
# Bài 2: điền kết quả tính tay
hand_answer = None
hand_pair = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành ngăn xếp đơn điệu và quét ngược.

In [27]:
def exercise_3_stack(nums: List[int]) -> int:
    stack = []
    for i, x in enumerate(nums):
        # TODO 1: đẩy i vào ngăn xếp nếu x nhỏ hơn nghiêm ngặt nums[stack[-1]] (hoặc ngăn xếp rỗng)
        pass
    ans = 0
    for j in range(len(nums) - 1, -1, -1):
        # TODO 2: while đỉnh ngăn xếp có giá trị <= nums[j]: cập nhật ans và pop
        pass
    return ans


# for nums, expected in examples:
#     assert exercise_3_stack(nums) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_sort(nums)` bằng sắp xếp chỉ số theo `(giá trị, chỉ số)`.

In [28]:
def exercise_4_sort(nums: List[int]) -> int:
    # TODO
    pass


# for nums, expected in examples:
#     assert exercise_4_sort(nums) == expected

### Bài 5 — Mini challenge: trả về một cặp tối ưu (khó)

Viết `exercise_5_pair(nums)` trả về một cặp `(i, j)` (`i < j`, `nums[i] <= nums[j]`) có độ rộng **lớn nhất**; nếu không có dốc thì trả `None`. Có thể có nhiều đáp án; ta kiểm tra bằng hàm kiểm tra.

```python
exercise_5_pair([6, 0, 8, 2, 1, 5])  -> (1, 5)
exercise_5_pair([3, 2, 1])           -> None
```

In [29]:
def exercise_5_pair(nums: List[int]):
    # TODO
    pass


# p = exercise_5_pair([6, 0, 8, 2, 1, 5])
# assert p is not None and p[0] < p[1] and [6, 0, 8, 2, 1, 5][p[0]] <= [6, 0, 8, 2, 1, 5][p[1]] and p[1] - p[0] == 4
# assert exercise_5_pair([3, 2, 1]) is None

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: kỷ lục thấp là phần tử nhỏ hơn nghiêm ngặt mọi phần tử trước.
- Mức 2: `[6,0,8,2,1,5]`: chỉ số `0` (6), `1` (0); `[1,2,3]`: chỉ `0`.
- Mức 3: `[0, 1]`, `[0]`, `[1, 3, 0, 2]`.

**Bài 2**
- Mức 1: kỷ lục thấp của `[5,3,4,1,2,6]`: chỉ số `0` (5), `1` (3), `3` (1).
- Mức 2: `j = 5` (giá trị 6) ăn cả ba: ưu tiên chỉ số nhỏ nhất `0`.
- Mức 3: độ rộng `5`, cặp `(0, 5)`.

**Bài 3**
- Mức 1: `if not stack or x < nums[stack[-1]]: stack.append(i)`.
- Mức 2: `while stack and nums[stack[-1]] <= nums[j]:`.
- Mức 3: `ans = max(ans, j - stack.pop())` bên trong vòng `while`.

**Bài 4**
- Mức 1: `order = sorted(range(n), key=lambda i: (nums[i], i))`.
- Mức 2: duyệt theo `order`, giữ `min_index`.
- Mức 3: `ans = max(ans, j - min_index)` rồi `min_index = min(min_index, j)`.

**Bài 5**
- Mức 1: như Cách 2 nhưng ghi lại cặp khi cập nhật `ans`.
- Mức 2: so sánh `j - i > best` mới cập nhật.
- Mức 3: trả `None` nếu `best == 0`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [30]:
print(record_lows([6, 0, 8, 2, 1, 5]))
print(record_lows([1, 2, 3]))
print(sorted(range(4), key=lambda i: ([3, 1, 3, 2][i], i)))

assert record_lows([6, 0, 8, 2, 1, 5]) == [0, 1]
assert record_lows([1, 2, 3]) == [0]
assert sorted(range(4), key=lambda i: ([3, 1, 3, 2][i], i)) == [1, 3, 0, 2]

[0, 1]
[0]
[1, 3, 0, 2]


### Đáp án Bài 2

Dốc `(0, 5)`: `5 <= 6`, độ rộng `5`.

In [31]:
nums = [5, 3, 4, 1, 2, 6]
hand_answer_solution = 5
hand_pair_solution = (0, 5)
assert Solution().maxWidthRamp(list(nums)) == hand_answer_solution == ramp_brute(nums)
i, j = hand_pair_solution
assert i < j and nums[i] <= nums[j] and j - i == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [32]:
def exercise_3_stack_solution(nums: List[int]) -> int:
    stack = []
    for i, x in enumerate(nums):
        if not stack or x < nums[stack[-1]]:
            stack.append(i)
    ans = 0
    for j in range(len(nums) - 1, -1, -1):
        while stack and nums[stack[-1]] <= nums[j]:
            ans = max(ans, j - stack.pop())
    return ans

for nums, expected in examples:
    assert exercise_3_stack_solution(nums) == expected
rng = random.Random(3)
for _ in range(300):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(2, 20))]
    assert exercise_3_stack_solution(nums) == ramp_brute(nums)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [33]:
def exercise_4_sort_solution(nums: List[int]) -> int:
    order = sorted(range(len(nums)), key=lambda i: (nums[i], i))
    min_index, ans = len(nums), 0
    for j in order:
        ans = max(ans, j - min_index)
        min_index = min(min_index, j)
    return ans

for nums, expected in examples:
    assert exercise_4_sort_solution(nums) == expected
rng = random.Random(4)
for _ in range(300):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(2, 20))]
    assert exercise_4_sort_solution(nums) == ramp_brute(nums)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [34]:
def exercise_5_pair_solution(nums: List[int]):
    stack = []
    for i, x in enumerate(nums):
        if not stack or x < nums[stack[-1]]:
            stack.append(i)
    best, pair = 0, None
    for j in range(len(nums) - 1, -1, -1):
        while stack and nums[stack[-1]] <= nums[j]:
            i = stack.pop()
            if j - i > best:
                best, pair = j - i, (i, j)
    return pair

def check_pair(nums, pair):
    b = ramp_brute(nums)
    if b == 0:
        return pair is None
    i, j = pair
    return 0 <= i < j < len(nums) and nums[i] <= nums[j] and j - i == b

assert check_pair([6, 0, 8, 2, 1, 5], exercise_5_pair_solution([6, 0, 8, 2, 1, 5]))
assert exercise_5_pair_solution([3, 2, 1]) is None
rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(2, 20))]
    assert check_pair(nums, exercise_5_pair_solution(nums)), nums
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (ngăn xếp đơn điệu + quét ngược)

```python
class Solution:
    def maxWidthRamp(self, nums: List[int]) -> int:
        stack = []                                   # chỉ số các kỷ lục thấp (giá trị giảm nghiêm ngặt)
        for i, x in enumerate(nums):
            if not stack or x < nums[stack[-1]]:
                stack.append(i)
        ans = 0
        for j in range(len(nums) - 1, -1, -1):       # quét j từ phải sang trái
            while stack and nums[stack[-1]] <= nums[j]:
                ans = max(ans, j - stack.pop())      # ăn mọi ứng viên thỏa
            if not stack:
                break
        return ans
```

Công thức nhớ nhanh: `i` chỉ là kỷ lục thấp; quét `j` ngược; `while nums[stack[-1]] <= nums[j]: ans = max(ans, j - pop)`.

### Checklist trước khi submit

- [ ] Ngăn xếp chỉ chứa **kỷ lục thấp** (đẩy khi `nums[i] < nums[stack[-1]]`)?
- [ ] So sánh `<=` khi `pop` (không `<`)?
- [ ] Dùng `while` (không `if`) để ăn nhiều ứng viên?
- [ ] Trả `0` khi không có dốc?
- [ ] Đã thử: giảm dần, tăng dần, toàn bằng nhau, nhiều trùng lặp, `n = 5 * 10^4`?
- [ ] Tên method đúng `maxWidthRamp(self, nums)` và trả `int`?